# FranchiseOps AI · Copilot fine-tune (intents, synonyms, "I do not know")

This notebook contains **only my part** of the project, as patches you can apply on top of `FranchiseOps_AI_Final_Code.ipynb`:

1. **New intents** in `INTENTS` of `copilot_pro.py` (pattern → title → SQL): profit and loss, sales and transactions today/yesterday/week/month, average order value, out-of-stock, outlets low on stock, running/ended campaigns, best channel by ROI, failed audits and pass rate, absentees today, headcount, unhappy staff, understaffed outlets, lapsed customers, ratings, tickets by severity/type/overdue/awaiting approval, city and tier rankings.
2. **New synonyms** in `qa.py` (column words, table words, and the `customers_fr` table, which the Copilot could not see before).
3. **Every number comes from SQL.** An intent only holds a SQL string; nothing is typed in or calculated in Python. "Profit" is defined in each title as revenue minus operating costs, because that is all the outlets table holds.
4. **When unsure, the Copilot says "I do not know"**: (a) questions the data can never answer (weather, competitors, suppliers, item-level sales, gender/contact data, GST, rent, forecasts, promotions) get a direct reply that says why; (b) a generic answer that shares no key word with the question is no longer shown, the Copilot says "I do not know" and logs the question in the backlog.
5. **Admin → Copilot insights** shows the answer rate before and after, plus the questions that still fail.

## How "answer rate" is measured (so the numbers mean something)
The old engine often returns *a* table that is not the answer (for example "below 3.5 customer satisfaction" returned the open-tickets list, because "satisfaction" contains "action"). So I measure two things: **answer rate** (an on-topic exact SQL answer was returned) and **verified correct** (the SQL or the numbers shown match an independent ground-truth query on the live database). 109 answerable and 31 out-of-scope questions are replayed through the real layers (`copilot_pro.prepare` → `analyst.py` → `qa.py`, no GPU, no LLM). "Before" runs the original files, "after" the tuned files, on the same database.

Results from my test run (simulated copy of the franchise data, 2026-11-11). **Your numbers will differ a little** because the data and the clock differ; run cell 5 to get yours.

| Question set | Answerable | Answer rate before → after | Verified correct before → after | Says "I do not know" when it should |
|---|---|---|---|---|
| **Tuning** (used to write the patterns) | 56 | 64% → 100% | 27% → 100% | 7 of 15 → 15 of 15 |
| **Paraphrases** (written first; I knew the wording while tuning) | 30 | 37% → 100% | 0% → 100% | 4 of 8 → 7 of 8 |
| **Blind** (written after tuning was frozen, run once) | 23 | 57% → 83% | 30% → 83% | 1 of 8 → 5 of 8 |
| **All** | 109 | 55% → 96% | 20% → 96% | 12 of 31 → 27 of 31 |


**Read the blind row as the honest one.** The tuning set was used to write the patterns, so 100% there is expected. The paraphrase set was written before tuning but I knew its wording while writing patterns, so it is not blind. The blind set was written afterwards and run once (first run 78%; one later general fix to the "weak match" test brought it to 83%).

## Where each cell goes in the main notebook
Run order: main notebook cells as they are → **cells 1–4 here** (after the cells that write `qa.py`, `copilot_pro.py`, `views_misc.py`, `server.py`, and before "Start the app") → start the app → **cell 5 here**. Every patch checks its anchors and stops with a clear message if the file differs, so nothing is half-applied silently.

## 1 · Patch `qa.py` (synonyms)
Run after the cell that writes `qa.py`, before the app starts. Safe to run twice.

In [ ]:
import os, re, sys
APP = "/content/app"
P = os.path.join(APP, "qa.py"); src = open(P, encoding="utf-8").read()
MARK = "# ===== COPILOT FINE-TUNE (qa.py) ====="
if MARK in src: print("qa.py: tuning already applied, nothing to do")
else:
    os.makedirs(os.path.join(APP, "_orig"), exist_ok=True)
    if not os.path.exists(os.path.join(APP, "_orig", "qa.py")): open(os.path.join(APP, "_orig", "qa.py"), "w", encoding="utf-8").write(src)   # kept for the 'before' measurement
    def rep(old, new):
        global src
        assert src.count(old) == 1, "qa.py anchor not found exactly once (the file differs from the reference notebook):\n" + old[:90]
        src = src.replace(old, new)
    # 1) 'customer satisfaction / rating / feedback' mean the outlet metric, not the customers table
    rep("    d = lookup(kind, q, scope)\n    if d: return d\n    T = TABLES[kind]; sc = {}",
        "    d = lookup(kind, q, scope)\n    if d: return d\n    ql = _normalise(ql)                                   # [tuning]\n    T = TABLES[kind]; sc = {}")
    # 2) 'open' tickets = open + awaiting approval ('proposed'), the same definition the briefing and the tickets page use
    rep("    if re.search(r\"\\b(open|pending|unresolved)\\b\", ql) and \"status\" in cs and not any(\"status\" in x for x in desc) and t in (\"agent_actions\", \"purchase_orders\"):",
        "    if t == \"agent_actions\" and \"status=?\" in where:                  # [tuning] 'open tickets' must count open AND proposed, as everywhere else in the app\n        _k = where.index(\"status=?\"); _p = sum(w.count(\"?\") for w in where[:_k])\n        if params[_p] == \"open\": where[_k] = \"status IN ('open','proposed')\"; del params[_p]\n    if re.search(r\"\\b(open|pending|unresolved)\\b\", ql) and \"status\" in cs and not any(\"status\" in x for x in desc) and t in (\"agent_actions\", \"purchase_orders\"):")
    src += '''

''' + MARK + '''
# Extra column synonyms, table synonyms and one new table (customers_fr). Edit the three dictionaries to teach the Copilot new words; nothing else needs to change.
TUNED_COLUMN_SYNONYMS = {
    "revenue": ["income", "sales value"],
    "customer_satisfaction": ["satisfaction score", "customer happiness", "happiness"],
    "staff_headcount": ["head count", "team size", "manpower"],
    "salary": ["compensation", "paid", "remuneration", "ctc"],
    "overtime_hrs": ["extra hours", "over time", "long hours", "overworked"],
    "job_satisfaction": ["morale", "engagement", "employee satisfaction", "staff satisfaction", "unhappy"],
    "work_life_balance": ["work life balance", "work-life balance", "wlb"],
    "tenure_years": ["years of service", "seniority", "length of service", "longest serving"],
    "predicted_attrition_prob": ["turnover risk", "flight risk", "attrition risk", "retention risk", "resign", "resignation"],
    "current_stock": ["in stock", "stock on hand", "stock quantity", "on hand"],
    "reorder_threshold": ["reorder point", "re-order level", "minimum stock", "safety stock", "par level"],
    "weekly_demand": ["weekly demand", "consumption", "usage", "fast moving", "fast-moving"],
    "lead_time_days": ["delivery time", "supplier lead time"],
    "stockout_risk_prob": ["stockouts", "stock-outs", "running out", "low on stock"],
    "actual_roi": ["return on investment", "roas", "campaign return", "marketing return", "payback"],
    "conversions": ["conversion count", "sign ups", "signups", "redemptions"],
    "budget": ["marketing budget", "ad spend", "promo spend", "campaign cost", "campaign spend"],
    "reach": ["impressions", "audience", "people reached"],
    "sentiment_score": ["mood", "positivity", "tone"],
    "rating": ["star rating", "average rating", "customer rating", "customer ratings"],
    "score": ["audit rating", "inspection score", "compliance score", "hygiene score", "safety score"],
    "violations": ["breach", "breaches", "non-compliance", "infractions", "findings"],
    "amount": ["sales amount"],
    "qty": ["order quantity", "units ordered", "quantity ordered"],
    "operating_costs": ["operating expense", "operating expenses", "expenses", "opex", "running cost", "running costs", "overheads", "expenditure"],
    "days_since_last_visit": ["days since last visit", "last visit", "recency", "since last visit", "inactive days"],
    "total_spend_90d": ["customer spend", "total spend", "spending", "spent", "spenders", "lifetime spend", "90 day spend"],
    "visit_freq": ["visit frequency", "visits", "how often", "frequency"],
    "avg_order_value": ["order value", "basket size", "basket", "aov", "average bill", "bill size"],
}
TUNED_TABLE_SYNONYMS = {"Franchise": {
    "outlets": ["shop", "restaurant", "cafe"],
    "staff": ["personnel", "manpower", "team member", "hire", "colleague", "manager", "store manager", "shift lead", "cashier", "cook", "chef", "crew", "inventory associate"],
    "inventory": ["stock", "ingredient", "supplies", "supply", "material", "consumable", "packaging"],
    "marketing": ["promo", "offer", "ads", "social media", "outreach"],
    "feedback": ["star", "survey", "testimonial", "opinion"],
    "audits": ["safety check", "hygiene check", "food safety"],
    "sales": ["billing", "bills", "bill", "pos", "till"],
    "purchase_orders": ["restock order", "supplier order", "replenishment", "stock order", "incoming stock"],
    "attendance": ["punctuality", "check-in", "clock in", "shift", "absentee", "no-show", "late arrival", "turn up"],
    "agent_actions": ["issue", "incident", "escalation", "recommendation", "to-do", "todo"],
}}
TUNED_TABLES = {"Franchise": {      # table: (synonyms, id column, columns shown, default ranking metric)
    "customers_fr": (["customer", "guest", "shopper", "diner", "patron", "loyalty member"], "customer_id", ["customer_id", "outlet_id", "total_spend_90d", "visit_freq", "avg_order_value", "days_since_last_visit"], "total_spend_90d"),
}}
def _apply_tuning():
    for c, extra in TUNED_COLUMN_SYNONYMS.items():
        CSYN[c] = CSYN.get(c, []) + [s for s in extra if s not in CSYN.get(c, [])]
    for kind, tabs in TUNED_TABLE_SYNONYMS.items():
        for t, extra in tabs.items():
            if t in TABLES.get(kind, {}):
                syn, idc, cols, met = TABLES[kind][t]; TABLES[kind][t] = (syn + [s for s in extra if s not in syn], idc, cols, met)
    for kind, tabs in TUNED_TABLES.items():
        for t, spec in tabs.items(): TABLES.setdefault(kind, {}).setdefault(t, spec)
    BAD.add("days_since_last_visit"); PL.setdefault("customers_fr", "customers")
_apply_tuning()
def _normalise(ql):
    """'customer satisfaction', 'customer rating', 'customer feedback' are outlet metrics: drop the word 'customer' so it does not select the customers table."""
    return re.sub(r"\\bcustomers?['’]?s? (satisfaction|ratings?|reviews?|feedback|complaints?|comments?|sentiment)\\b", r"\\1", ql)
'''
    open(P, "w", encoding="utf-8").write(src); print("qa.py: tuning applied")

## 2 · Patch `copilot_pro.py` (intents, "I do not know")
Run after the cell that writes `copilot_pro.py`. The new patterns are in the block `TUNED_INTENTS` / `TUNED_UNSUPPORTED` inside that file after this cell runs.

In [ ]:
import os, re, sys
APP = "/content/app"
BLOCK = r'''# ------------------------------------------------------------------ TUNED INTENTS (Copilot fine-tune). Tried BEFORE the original INTENTS below. Add a line to teach a new question:
#   (regex on the lower-cased question, answer title, SQL, needs_id [, message shown when the query returns no rows])
# Every number shown is the result of the SQL in the same line: nothing is calculated or typed in here. needs_id=True binds the ID found in the question to the single '?'.
_NO_ID = r"(?!.*\b(?:out|stf|emp|cmp|aud|fb|inv)-\d)"                                # leave questions about one ID to the ID intents
_NOT_BY = r"^(?!.*\b(?:by|per|each|which|top|best|highest|lowest|worst|rank\w*|outlets?|stores?|branch\w*)\b)"
_OUTLET = r"(?=.*\b(?:outlets?|stores?|branch\w*|shops?)\b)"
_SALES_W = r"(?:sales|sell|sold|selling|revenue|takings|turnover|earn\w*|income)"
_TX_W = r"(?:transactions?|bills?|invoices?|checkouts?|baskets?|footfall|orders?|receipts?|customers? (?:served|visited|came|walked in))"
_PERIODS = [                      # (regex, SQL condition on sales.ts, label). 'today' is the IST date, as everywhere in the app.
 (r"\blast month\b|\bprevious month\b", "strftime('%Y-%m', substr(ts, 1, 10)) = strftime('%Y-%m', 'now', 'localtime', 'start of month', '-1 month')", "last calendar month (IST)"),
 (r"\bthis month\b|\bmonth to date\b|\bmtd\b", "strftime('%Y-%m', substr(ts, 1, 10)) = strftime('%Y-%m', 'now', 'localtime')", "this month so far (IST)"),
 (r"\b(?:last|past) (?:30|thirty) days\b|\bpast month\b", "substr(ts, 1, 10) >= date('now', 'localtime', '-29 day')", "in the last 30 days (IST)"),
 (r"\btoday\b|\bso far today\b|\bthis (?:morning|afternoon)\b", "substr(ts, 1, 10) = date('now', 'localtime')", "today (IST)"),
 (r"\byesterday\b", "substr(ts, 1, 10) = date('now', 'localtime', '-1 day')", "yesterday (IST)"),
 (r"\b(?:last|past|previous) (?:7|seven) days\b|\b(?:this|last|past) week\b|\bweek\b", "substr(ts, 1, 10) >= date('now', 'localtime', '-6 day')", "in the last 7 days (IST)"),
]
def _sales_intents():
    out = []
    for rx, cond, lab in _PERIODS:
        out.append((_NOT_BY + _NO_ID + r"(?=.*\b" + _SALES_W + r"\b)(?=.*(?:" + rx + r"))", "Total sales " + lab + ", all outlets.",
                    "SELECT ROUND(SUM(amount), 0) AS sales, SUM(tx_count) AS transactions, COUNT(DISTINCT outlet_id) AS outlets_reporting FROM sales WHERE " + cond + " HAVING COUNT(*) > 0", False))
    for rx, cond, lab in _PERIODS:
        out.append((r"^(?!.*\b(?:purchase|supplier|reorder|re-order|stock|per|by|each|which|top|best|highest|lowest)\b)" + _NO_ID + r"(?=.*\b" + _TX_W + r"\b)(?=.*(?:" + rx + r"))", "Transactions " + lab + ", all outlets.",
                    "SELECT SUM(tx_count) AS transactions, ROUND(SUM(amount), 0) AS sales, COUNT(DISTINCT outlet_id) AS outlets_reporting FROM sales WHERE " + cond + " HAVING COUNT(*) > 0", False))
    for asc in (True, False):
        for rx, cond, lab in _PERIODS:
            words = r"(?:lowest|least|worst|weakest|smallest|bottom)" if asc else r"(?:which|top|best|highest|most|biggest|largest|rank\w*|by outlet|per outlet|each outlet)"
            out.append((_NO_ID + r"(?=.*\b" + words + r"\b)" + (r"" if asc else r"(?!.*\b(?:lowest|least|worst|weakest|smallest|bottom)\b)") + r"(?=.*\b" + _SALES_W + r"\b)(?=.*(?:" + rx + r"))", "Sales " + lab + " by outlet, " + ("lowest" if asc else "highest") + " first.",
                        "SELECT outlet_id, ROUND(SUM(amount), 0) AS sales, SUM(tx_count) AS transactions FROM sales WHERE " + cond + " GROUP BY outlet_id ORDER BY sales" + ("" if asc else " DESC") + ", outlet_id LIMIT 10", False))
    return out
TUNED_INTENTS = [
 # ---- profit and cost. The outlets table holds revenue and operating_costs only, so 'profit' = revenue - operating costs (stated in every title).
 (r"\b(?:margin|margins)\b", "Profit margin of {id} (operating profit divided by revenue, %):", "SELECT ROUND(100.0 * (revenue - operating_costs) / revenue, 1) FROM outlets WHERE outlet_id=?", True),
 (r"\b(?:profit|profits|profitable|earnings|net income|bottom line)\b", "Operating profit of {id} (revenue minus operating costs):", "SELECT CAST(ROUND(revenue - operating_costs, 0) AS INT) FROM outlets WHERE outlet_id=?", True),
 (_NO_ID + _OUTLET + r"(?=.*\b(?:loss|losses|loss[- ]making|losing money|unprofitable|not profitable|in the red|negative profit)\b)", "Outlets whose operating costs exceed their revenue (loss-making).",
  "SELECT outlet_id, outlet_name, location, ROUND(revenue, 0) AS revenue, ROUND(operating_costs, 0) AS operating_costs, ROUND(revenue - operating_costs, 0) AS profit FROM outlets WHERE operating_costs > revenue ORDER BY profit LIMIT 15", False,
  "No outlet is loss-making: the operating costs of every outlet are below its revenue."),
 (_NO_ID + r"(?=.*\b(?:lowest|least|worst|weakest|smallest|bottom|poorest)\b)(?=.*\bmargins?\b)", "Outlets with the lowest profit margin (operating profit divided by revenue).",
  "SELECT outlet_id, outlet_name, location, ROUND(100.0 * (revenue - operating_costs) / revenue, 1) AS margin_pct, ROUND(revenue - operating_costs, 0) AS profit, ROUND(revenue, 0) AS revenue FROM outlets ORDER BY margin_pct, outlet_id LIMIT 10", False),
 (_NO_ID + r"(?=.*\bmargins?\b)", "Outlets with the highest profit margin (operating profit divided by revenue).",
  "SELECT outlet_id, outlet_name, location, ROUND(100.0 * (revenue - operating_costs) / revenue, 1) AS margin_pct, ROUND(revenue - operating_costs, 0) AS profit, ROUND(revenue, 0) AS revenue FROM outlets ORDER BY margin_pct DESC, outlet_id LIMIT 10", False),
 (_NO_ID + r"(?=.*\b(?:lowest|least|worst|weakest|smallest|bottom|poorest)\b)(?=.*\b(?:profit\w*|margins?)\b)", "Outlets with the lowest operating profit (revenue minus operating costs).",
  "SELECT outlet_id, outlet_name, location, ROUND(revenue, 0) AS revenue, ROUND(operating_costs, 0) AS operating_costs, ROUND(revenue - operating_costs, 0) AS profit, ROUND(100.0 * (revenue - operating_costs) / revenue, 1) AS margin_pct FROM outlets ORDER BY profit LIMIT 10", False),
 (_NO_ID + r"(?=.*\b(?:profit\w*|margins?|earnings)\b)", "Outlets ranked by operating profit (revenue minus operating costs).",
  "SELECT outlet_id, outlet_name, location, ROUND(revenue, 0) AS revenue, ROUND(operating_costs, 0) AS operating_costs, ROUND(revenue - operating_costs, 0) AS profit, ROUND(100.0 * (revenue - operating_costs) / revenue, 1) AS margin_pct FROM outlets ORDER BY profit DESC LIMIT 10", False),
 (_NO_ID + r"(?=.*\b(?:revenue|sales)\b.*\bper (?:employee|staff|head|person|worker|team member)\b|.*\bproductivity\b)", "Revenue per staff member by outlet (outlet revenue divided by its headcount).",
  "SELECT outlet_id, location, ROUND(revenue, 0) AS revenue, staff_headcount, ROUND(revenue * 1.0 / staff_headcount, 0) AS revenue_per_staff FROM outlets WHERE staff_headcount > 0 ORDER BY revenue_per_staff DESC LIMIT 10", False),
 # ---- sales and transactions (company level; per-outlet questions still use the original intents)
 *_sales_intents(),
 (r"\b(?:which|highest|lowest|top|best|worst|by outlet|per outlet|each outlet)\b.*\b(?:average|avg|mean)\b.*\b(?:basket|bill|order|ticket|transaction)\b|\b(?:average|avg|mean)\b.*\b(?:basket|bill|order|ticket|transaction)\b.*\b(?:by outlet|per outlet|each outlet|which outlet)\b", "Average order value by outlet (sales divided by transactions), highest first.",
  "SELECT outlet_id, ROUND(SUM(amount) / SUM(tx_count), 2) AS avg_order_value, SUM(tx_count) AS transactions, ROUND(SUM(amount), 0) AS sales FROM sales WHERE tx_count > 0 GROUP BY outlet_id ORDER BY avg_order_value DESC LIMIT 10", False),
 (r"^(?!.*\b(?:purchase|supplier|reorder|stock)\b)(?=.*\b(?:average|avg|mean|typical|usual)\b.*\b(?:basket|bill|order|ticket|transaction)\b|.*\baov\b|.*\bbasket size\b)", "Average order value (sales amount divided by transactions), over all sales records.",
  "SELECT ROUND(SUM(amount) / SUM(tx_count), 2) AS avg_order_value, SUM(tx_count) AS transactions, ROUND(SUM(amount), 0) AS sales, MIN(substr(ts, 1, 10)) AS from_date, MAX(substr(ts, 1, 10)) AS to_date FROM sales WHERE tx_count > 0", False),
 # ---- stock
 (r"\b(?:out of stock|sold out|zero stock|no stock|stocked out|stock[- ]?outs?(?! risk)|empty shelves|completely out)\b", "Items at zero stock right now.",
  "SELECT outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days FROM inventory WHERE current_stock <= 0 ORDER BY weekly_demand DESC, outlet_id LIMIT 20", False, "No item is at zero stock right now."),
 (r"\b(?:outlets?|stores?|branch\w*|shops?)\b.*\b(?:low[- ]stock|below (?:the )?reorder|under (?:the )?reorder|at or below|restock\w*|re-?order\w*|running low|short of stock|low on stock)\b|\b(?:most|highest|largest)\b.*\b(?:low[- ]stock|below reorder)\b", "Outlets with the most SKUs at or below their reorder level.",
  "SELECT outlet_id, SUM(current_stock <= reorder_threshold) AS skus_at_or_below_reorder, COUNT(*) AS skus, SUM(current_stock <= 0) AS out_of_stock FROM inventory GROUP BY outlet_id ORDER BY skus_at_or_below_reorder DESC, outlet_id LIMIT 10", False),
 # ---- campaigns (marketing.start_date / end_date against today's IST date)
 (r"\b(?:how many|number of|count of)\b.*\b(?:active|running|live|ongoing|current)\b.*\b(?:campaigns?|promotions?|promos?)\b|\b(?:how many)\b.*\b(?:campaigns?|promotions?|promos?)\b.*\b(?:running|live|active|ongoing)\b", "Campaigns running today (start date on or before today, end date on or after today).",
  "SELECT COUNT(*) AS running_campaigns, ROUND(SUM(budget), 0) AS budget, SUM(conversions) AS conversions FROM marketing WHERE start_date <= date('now', 'localtime') AND end_date >= date('now', 'localtime')", False),
 (r"\b(?:active|running|live|ongoing|current|in progress|at the moment|right now)\b.*\b(?:campaigns?|promotions?|promos?|offers?)\b|\b(?:campaigns?|promotions?|promos?)\b.*\b(?:running|live|active|ongoing|in progress)\b", "Campaigns running today (start date on or before today, end date on or after today).",
  "SELECT campaign_id, campaign_name, outlet_id, channel, budget, actual_roi, start_date, end_date FROM marketing WHERE start_date <= date('now', 'localtime') AND end_date >= date('now', 'localtime') ORDER BY end_date, campaign_id LIMIT 25", False, "No campaign is running today."),
 (r"\b(?:ended|expired|finished|completed|closed|already over|no longer running|stopped)\b.*\b(?:campaigns?|promotions?|promos?)\b|\b(?:campaigns?|promotions?|promos?)\b.*\b(?:ended|expired|finished|completed|are over|have stopped)\b", "Campaigns that have ended (latest first).",
  "SELECT campaign_id, campaign_name, outlet_id, channel, budget, actual_roi, end_date, (SELECT COUNT(*) FROM marketing WHERE end_date < date('now', 'localtime')) AS total_ended FROM marketing WHERE end_date < date('now', 'localtime') ORDER BY end_date DESC, campaign_id LIMIT 15", False, "No campaign has ended yet."),
 (r"\b(?:worst|lowest|weakest|poorest|least)\b.*\b(?:channels?|platforms?|mediums?)\b", "Marketing channels by average ROI, weakest first.",
  "SELECT channel, COUNT(*) AS campaigns, ROUND(AVG(actual_roi), 2) AS avg_roi, ROUND(SUM(budget), 0) AS total_budget, SUM(conversions) AS conversions FROM marketing GROUP BY channel ORDER BY avg_roi ASC", False),
 (r"\b(?:channels?|platforms?|mediums?)\b.*\b(?:roi|return|best|perform\w*|effective\w*|budget|spend|conversion\w*)\b|\b(?:roi|return|budget|spend|conversions?)\b.*\b(?:by|per|each|across)\b.*\b(?:channels?|platforms?|mediums?)\b|\bbest (?:marketing |advertising )?(?:channel|platform)\b", "Marketing channels by average ROI, best first.",
  "SELECT channel, COUNT(*) AS campaigns, ROUND(AVG(actual_roi), 2) AS avg_roi, ROUND(SUM(budget), 0) AS total_budget, SUM(conversions) AS conversions FROM marketing GROUP BY channel ORDER BY avg_roi DESC", False),
 # ---- audits (status values in the data: Pass, Warning, Conditional Pass, Action Required, Fail)
 (r"\b(?:pass rate|passing rate|pass percentage)\b|\b(?:percent\w*|share|proportion)\b.*\b(?:audits?|inspections?)\b.*\bpass\w*\b", "Audit pass rate (share of audits with status Pass).",
  "SELECT ROUND(100.0 * SUM(status = 'Pass') / COUNT(*), 1) AS pass_rate_pct, SUM(status = 'Pass') AS passed, COUNT(*) AS audits, SUM(status IN ('Fail', 'Action Required')) AS failed_or_action_required FROM audits", False),
 (r"\b(?:fail\w*|not pass\w*|non[- ]?complian\w*|action required)\b.*\b(?:audits?|inspections?)\b|\b(?:audits?|inspections?)\b.*\b(?:fail\w*|non[- ]?complian\w*|action required)\b", "Audits that failed or need action (status Fail or Action Required), newest first.",
  "SELECT audit_id, outlet_id, audit_date, score, violations, category, status FROM audits WHERE status IN ('Fail', 'Action Required') ORDER BY audit_date DESC, score, audit_id LIMIT 20", False, "No audit has the status Fail or Action Required."),
 (r"\baudits?\b.*\b(?:by|per)\b.*\bstatus\b|\baudit (?:results?|outcomes?|summary|breakdown)\b", "Audit results by status.",
  "SELECT status, COUNT(*) AS audits, ROUND(AVG(score), 1) AS avg_score, SUM(violations) AS violations FROM audits GROUP BY status ORDER BY audits DESC", False),
 (_OUTLET + r"(?=.*\b(?:audit|inspection)s?\b)(?=.*\b(?:best|highest|top|strongest)\b)(?=.*\b(?:score|scores|rated)\b)", "Outlets with the highest average audit score.",
  "SELECT outlet_id, ROUND(AVG(score), 1) AS avg_score, COUNT(*) AS audits, SUM(violations) AS violations FROM audits GROUP BY outlet_id ORDER BY avg_score DESC, outlet_id LIMIT 10", False),
 (_OUTLET + r"(?=.*\b(?:audit|inspection)s?\b)(?=.*\b(?:score|scores|lowest|worst|weakest|poorest)\b)", "Outlets with the lowest average audit score.",
  "SELECT outlet_id, ROUND(AVG(score), 1) AS avg_score, COUNT(*) AS audits, SUM(violations) AS violations FROM audits GROUP BY outlet_id ORDER BY avg_score, outlet_id LIMIT 10", False),
 (r"\b(?:likely|going|about|expected|predicted|planning|plan) to (?:leave|quit|resign)\b|\bwho (?:will|might|may) (?:leave|quit|resign)\b", "Staff most likely to leave (highest predicted attrition risk).",
  "SELECT staff_id, name, role, outlet_id, predicted_attrition_prob, overtime_hrs, salary FROM staff ORDER BY predicted_attrition_prob DESC LIMIT 10", False),
 # ---- staff and attendance (attendance rows are the check-ins logged today; counts are distinct staff)
 (r"(?=.*\b(?:outlets?|stores?|branch\w*|which|by outlet)\b)(?=.*\b(?:absent|absentees?|absence|no[- ]?show|called in sick)\b)(?=.*\b(?:today|this morning|now)\b)", "Absent and late staff today (IST) by outlet, among the attendance records logged today.",
  "SELECT outlet_id, COUNT(DISTINCT CASE WHEN status = 'Absent' THEN staff_id END) AS absent_staff, COUNT(DISTINCT CASE WHEN status = 'Late' THEN staff_id END) AS late_staff, COUNT(DISTINCT staff_id) AS staff_logged FROM attendance WHERE substr(ts, 1, 10) = date('now', 'localtime') GROUP BY outlet_id ORDER BY absent_staff DESC, outlet_id LIMIT 10", False),
 (r"\b(?:absent|absentees?|absence|no[- ]?show|didn'?t turn up|did not turn up|called in sick)\b.*\b(?:today|this morning|now)\b|\b(?:today|this morning)\b.*\b(?:absent|absentees?|absence|no[- ]?show)\b", "Staff absent or late today (IST), among the attendance records logged today.",
  "SELECT COUNT(DISTINCT CASE WHEN status = 'Absent' THEN staff_id END) AS absent_staff, COUNT(DISTINCT CASE WHEN status = 'Late' THEN staff_id END) AS late_staff, COUNT(DISTINCT staff_id) AS staff_logged FROM attendance WHERE substr(ts, 1, 10) = date('now', 'localtime') HAVING COUNT(*) > 0", False),
 (r"^(?=.*\b(?:outlets?|stores?|branch\w*|shops?)\b)(?=.*\b(?:fewest|smallest|least|lowest)\b)(?=.*\b(?:staff|employees?|headcount|people|workers|manpower|team)\b)", "Outlets with the smallest declared headcount (outlets.staff_headcount); staff rows on record shown beside it.",
  "SELECT o.outlet_id, o.outlet_name, o.location, o.staff_headcount, (SELECT COUNT(*) FROM staff s WHERE s.outlet_id = o.outlet_id) AS staff_on_record FROM outlets o ORDER BY o.staff_headcount, o.outlet_id LIMIT 10", False),
 (r"^(?=.*\b(?:outlets?|stores?|branch\w*|shops?)\b)(?=.*\b(?:most|largest|biggest|highest|greatest|employs?)\b)(?=.*\b(?:staff|employees?|headcount|people|workers|manpower|team)\b)", "Outlets with the largest declared headcount (outlets.staff_headcount); staff rows on record shown beside it.",
  "SELECT o.outlet_id, o.outlet_name, o.location, o.staff_headcount, (SELECT COUNT(*) FROM staff s WHERE s.outlet_id = o.outlet_id) AS staff_on_record FROM outlets o ORDER BY o.staff_headcount DESC, o.outlet_id LIMIT 10", False),
 (r"\b(?:job satisfaction|unhappy|dissatisfied|disengaged|low morale|morale|work[- ]life balance|not happy)\b", "Staff with the lowest job satisfaction (ties broken by work-life balance, then overtime).",
  "SELECT staff_id, name, role, outlet_id, job_satisfaction, work_life_balance, overtime_hrs, predicted_attrition_prob FROM staff ORDER BY job_satisfaction, work_life_balance, overtime_hrs DESC, staff_id LIMIT 10", False),
 (r"\b(?:understaffed|under[- ]staffed|short[- ]?staffed|short of staff|staff(?:ing)? (?:shortage|gaps?|short)|not enough (?:staff|people|employees)|need more staff|lack of staff|shortage of staff)\b", "Outlets with an open staffing-shortage ticket (raised by the staff agent), critical first.",
  "SELECT target AS outlet_id, severity, status, detail FROM agent_actions WHERE type = 'staffing_shortage' AND status IN ('open', 'proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, target LIMIT 15", False, "No outlet has an open staffing-shortage ticket."),
 # ---- customers (customers_fr: one row per customer, spend over 90 days, visit frequency, days since the last visit). 30 days is the threshold used for 'lapsed'; other thresholds go to the generic engine.
 (r"^(?!.*\b(?!30\b)\d+\s*(?:days?|weeks?|months?)\b)(?=.*\b(?:outlets?|stores?|branch\w*|which|per outlet|by outlet)\b)(?=.*\b(?:lapsed|inactive|dormant|churned|stopped (?:visiting|coming))\b)(?=.*\b(?:customers?|guests?|shoppers?)\b)", "Outlets with the most lapsed customers (no visit in more than 30 days).",
  "SELECT outlet_id, COUNT(*) AS lapsed_customers, (SELECT COUNT(*) FROM customers_fr c2 WHERE c2.outlet_id = customers_fr.outlet_id) AS all_customers FROM customers_fr WHERE days_since_last_visit > 30 GROUP BY outlet_id ORDER BY lapsed_customers DESC, outlet_id LIMIT 10", False),
 (r"^(?!.*\b(?!30\b)\d+\s*(?:days?|weeks?|months?)\b)(?=.*\b(?:how many|number of|count)\b)(?=.*\b(?:customers?|guests?|shoppers?|diners?|patrons?)\b)(?=.*(?:\bnot (?:visited|returned|come|been back|come back)\b|\bhaven'?t (?:visited|returned|come|been back)\b|\blapsed\b|\binactive\b|\bdormant\b|\bchurned\b|\ba month\b|\b30 days\b))", "Customers with no visit in more than 30 days.",
  "SELECT COUNT(*) AS lapsed_customers, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customers_fr), 1) AS pct_of_all_customers, (SELECT COUNT(*) FROM customers_fr) AS all_customers FROM customers_fr WHERE days_since_last_visit > 30", False),
 # ---- feedback
 (r"\b(?:reviews?|ratings?|feedback)\b.*\b(?:by|per|each)\b.*\b(?:star|stars|rating)\b|\b(?:star rating|rating) (?:distribution|breakdown|split|mix)\b", "Reviews by star rating.",
  "SELECT rating, COUNT(*) AS reviews, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM feedback), 1) AS pct_of_reviews FROM feedback GROUP BY rating ORDER BY rating DESC", False),
 (r"\b(?:complain\w*|complaints?)\b.*\b(?:about|most|common|frequent|main|top|biggest)\b|\b(?:most (?:common|frequent)|top|main|biggest) (?:complaints?|issues|problems)\b|\bwhat are customers (?:saying|complaining)\b", "Most frequent comments on 1 and 2 star reviews.",
  "SELECT comment, COUNT(*) AS mentions, ROUND(AVG(rating), 2) AS avg_rating FROM feedback WHERE rating <= 2 GROUP BY comment ORDER BY mentions DESC, comment LIMIT 10", False),
 (_OUTLET + r"(?=.*\b(?:lowest|worst|poorest|weakest|bottom|least)\b)(?=.*\b(?:ratings?|reviews?|stars?|rated)\b)", "Outlets with the lowest average review rating (outlets with at least 3 reviews).",
  "SELECT outlet_id, ROUND(AVG(rating), 2) AS avg_rating, COUNT(*) AS reviews, ROUND(AVG(sentiment_score), 2) AS avg_sentiment FROM feedback GROUP BY outlet_id HAVING COUNT(*) >= 3 ORDER BY avg_rating, outlet_id LIMIT 10", False),
 (_OUTLET + r"(?=.*\b(?:best|highest|top|strongest)\b)(?=.*\b(?:ratings?|reviews?|stars?|rated)\b)", "Outlets with the highest average review rating (outlets with at least 3 reviews).",
  "SELECT outlet_id, ROUND(AVG(rating), 2) AS avg_rating, COUNT(*) AS reviews, ROUND(AVG(sentiment_score), 2) AS avg_sentiment FROM feedback GROUP BY outlet_id HAVING COUNT(*) >= 3 ORDER BY avg_rating DESC, outlet_id LIMIT 10", False),
 # ---- tickets (open = status open or proposed; 'proposed' means awaiting human approval)
 (r"\b(?:tickets?|alerts?|actions?|issues?|tasks?)\b.*\b(?:by|per|each|across)\b.*\bseverity\b|\bseverity\b.*\b(?:breakdown|split|summary|distribution)\b|\bhow many\b.*\b(?:critical|warning|info)\b.*\b(?:tickets?|alerts?|actions?|issues?)\b", "Open tickets by severity (open plus awaiting approval).",
  "SELECT severity, COUNT(*) AS tickets, SUM(status = 'open') AS open, SUM(status = 'proposed') AS awaiting_approval FROM agent_actions WHERE status IN ('open', 'proposed') GROUP BY severity ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END", False),
 (r"\b(?:tickets?|alerts?|actions?|issues?|tasks?)\b.*\b(?:by|per|each|across)\b.*\b(?:type|kind|category)\b|\bticket types?\b", "Open tickets by type (open plus awaiting approval).",
  "SELECT type, COUNT(*) AS tickets, SUM(severity = 'Critical') AS critical FROM agent_actions WHERE status IN ('open', 'proposed') GROUP BY type ORDER BY tickets DESC, type", False),
 (r"\boverdue\b|\bpast (?:their )?due\b|\b(?:tickets?|actions?|tasks?|alerts?|issues?)\b.*\b(?:breach\w*|missed)\b|\bsla (?:breach\w*|violations?|misses|missed)\b", "Open tickets that are past their due time (IST), oldest first.",
  "SELECT action_id, type, target, severity, status, due FROM agent_actions WHERE status IN ('open', 'proposed') AND due < strftime('%Y-%m-%dT%H:%M:%S', 'now', 'localtime') ORDER BY due, action_id LIMIT 15", False, "No open ticket is past its due time."),
 (r"\b(?:awaiting|pending|waiting (?:for )?|need(?:s|ing)?|requir\w+) (?:my |your |human |manager )?(?:approval|sign[- ]?off)\b|\bto approve\b|\bapprovals? (?:queue|pending|needed)\b|\b(?:what|which)\b.*\b(?:awaiting|pending) (?:approval|sign[- ]?off)\b", "Actions awaiting human approval (status 'proposed'), critical first.",
  "SELECT action_id, type, target, severity, detail, due FROM agent_actions WHERE status = 'proposed' ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, due, action_id LIMIT 15", False, "Nothing is awaiting approval."),
 # ---- outlets by city and by tier
 (r"\b(?:worst|lowest|weakest|poorest|least)\b.*\b(?:cit(?:y|ies)|towns?)\b.*\b(?:revenue|sales|turnover|performance|performs?|csat|satisfaction)\b|\b(?:cit(?:y|ies)|towns?)\b.*\b(?:lowest|worst|weakest)\b.*\b(?:revenue|sales)\b", "Cities by total revenue, lowest first.",
  "SELECT location AS city, COUNT(*) AS outlets, ROUND(SUM(revenue), 0) AS revenue, ROUND(AVG(customer_satisfaction), 2) AS avg_csat FROM outlets GROUP BY location ORDER BY revenue ASC LIMIT 12", False),
 (r"\b(?:which|what|top|best|highest|most|rank\w*)\b.*\b(?:cit(?:y|ies)|towns?)\b.*\b(?:revenue|sales|turnover|earns?|performance|performs?|csat|satisfaction)\b|\b(?:cit(?:y|ies)|towns?)\b.*\b(?:highest|top|best|most)\b.*\b(?:revenue|sales)\b", "Cities by total revenue, highest first.",
  "SELECT location AS city, COUNT(*) AS outlets, ROUND(SUM(revenue), 0) AS revenue, ROUND(AVG(customer_satisfaction), 2) AS avg_csat FROM outlets GROUP BY location ORDER BY revenue DESC LIMIT 12", False),
 (r"\b(?:worst|lowest|weakest|poorest|least)\b.*\b(?:tiers?|(?:store|outlet) formats?|formats? of (?:store|outlet)s?)\b|\b(?:tiers?|(?:store|outlet) formats?|formats? of (?:store|outlet)s?)\b.*\b(?:worst|lowest|weakest|poorly|least)\b", "Outlet tiers by total revenue, lowest first.",
  "SELECT tier, COUNT(*) AS outlets, ROUND(SUM(revenue), 0) AS revenue, ROUND(AVG(customer_satisfaction), 2) AS avg_csat, SUM(staff_headcount) AS staff FROM outlets GROUP BY tier ORDER BY revenue ASC", False),
 (r"\b(?:which|what|best|highest|top|strongest)\b.*\b(?:tiers?|(?:store|outlet) formats?|formats? of (?:store|outlet)s?)\b", "Outlet tiers by total revenue, highest first.",
  "SELECT tier, COUNT(*) AS outlets, ROUND(SUM(revenue), 0) AS revenue, ROUND(AVG(customer_satisfaction), 2) AS avg_csat, SUM(staff_headcount) AS staff FROM outlets GROUP BY tier ORDER BY revenue DESC", False),
]
# Questions the data can never answer: the Copilot says 'I do not know' and says why, instead of showing an unrelated table.
TUNED_UNSUPPORTED = [
 (r"\b(?:weather|rain|rainfall|monsoon)\b", "I do not know. The franchise database has no weather data, so I cannot say what the weather is or was."),
 (r"\b(?:competitors?|competition|competing)\b", "I do not know. The database holds only our own outlets; it has no competitor prices, sales or locations."),
 (r"\b(?:suppliers?|unit cost|cost price|purchase price|cogs|cost of goods|food cost|ingredient cost)\b", "I do not know. The data has no supplier names or prices: inventory and purchase orders hold quantities only. I can show items at or below reorder level or the purchase orders raised."),
 (r"\b(?:best|top|worst|slowest|fastest)[- ]?(?:selling|sellers?|moving)\b|\bunits?\b.*\b(?:sold|sell|selling)\b", "I do not know. Sales are stored as totals per outlet (amount and transaction count), not per product, so I cannot say which items sold or how many units."),
 (r"\b(?:gender|sex ratio|religion|nationality|marital|caste|ethnicity|date of birth|birthday|phone|mobile number|telephone|e-?mail address|home address|residential address)\b", "I do not know. Gender, contact details and addresses are not stored. Staff records hold only name, role, outlet, salary, age, tenure and satisfaction scores."),
 (r"^(?=.*\b(?:how much|how many|total|amount|number|count|rate|percentage|average|sum|share)\b)(?=.*\b(?:gst|vat|taxes|tax|upi|cash payments?|card payments?|payment (?:mode|method)s?|swiggy|zomato|uber eats|dunzo|delivery app|coupons?|discount codes?|loyalty points?|chargebacks?)\b)", "I do not know. The data has no tax, payment-method, delivery-app, coupon or loyalty information."),
 (r"^(?=.*\b(?:how much|total|amount|bill|bills|cost|costs|pay|paying|paid|monthly|per month|what is the|what's the)\b)(?=.*\b(?:rent|electricity|utilit(?:y|ies)|water bill|power bill|lease|royalt(?:y|ies)|franchise fees?)\b)", "I do not know. Rent, utilities, royalties and fees are not stored; the outlets table has only one operating-cost total per outlet."),
 (r"\b(?:next (?:year|month|quarter)|coming (?:year|month|quarter)|forecast\w*|predict\w*|projected|projection)\b.*\b(?:revenue|sales|profit|turnover|income|earnings)\b|\bwhat will (?:our |the )?(?:revenue|sales|profit)\b", "I do not know. I only report numbers that a SQL query returns from stored data, and a future figure is not stored. Ask for past sales, for example 'total sales in the last 7 days'."),
 (r"\b(?:promotion|promote|appraisal|performance (?:rating|review|score|appraisal)s?|deserves?|bonus(?:es)?|incentives?|commission|increment)\b", "I do not know. The staff data has no performance ratings, promotion history or bonus information. It holds salary, overtime, job satisfaction, tenure and attrition risk, so I can rank staff on those."),
]
'''
P = os.path.join(APP, "copilot_pro.py"); src = open(P, encoding="utf-8").read()
if "# [tuning]" in src: print("copilot_pro.py: tuning already applied, nothing to do")
else:
    os.makedirs(os.path.join(APP, "_orig"), exist_ok=True)
    if not os.path.exists(os.path.join(APP, "_orig", "copilot_pro.py")): open(os.path.join(APP, "_orig", "copilot_pro.py"), "w", encoding="utf-8").write(src)
    def rep(old, new):
        global src
        assert src.count(old) == 1, "copilot_pro.py anchor not found exactly once (file differs from the reference notebook):\n" + old[:100]
        src = src.replace(old, new)
    # 1) vocabulary so the typo repair does not mangle the new words
    rep("# ------------------------------------------------------------------ vocabulary / typo repair",
        'EXTRA_WORDS += """profit profits profitable loss losses margin margins basket transactions absent absence overdue severity tenure lapsed inactive dormant expired ended failed passed running active understaffed promotion promotions""".split()   # [tuning]\n# ------------------------------------------------------------------ vocabulary / typo repair')
    # 2) tuned intents and tuned 'I do not know' list in front of the original ones
    rep('INTENTS = {\n "Franchise": [\n', 'INTENTS = {\n "Franchise": TUNED_INTENTS + [\n')
    rep('UNSUPPORTED = {\n "Franchise": [', "# [tuning]\n" + BLOCK + '\nUNSUPPORTED = {\n "Franchise": TUNED_UNSUPPORTED + [')
    rep("The inventory data has no expiry dates, so I cannot answer that.", "I do not know: the inventory data has no expiry dates.")
    rep(r'[(r"\b(expired|expiry|expiring|shelf[- ]?life|best before)\b", ', r'[(r"^(?!.*\b(?:campaigns?|promotions?|promos?|offers?|audits?)\b).*\b(expired|expiry|expiring|shelf[- ]?life|best before)\b", ')   # 'expired campaigns' is not about stock expiry
    # 3) bugs in two original patterns: 'satisfaction' contains 'action', 'below' contains 'low'
    rep(r"(which|what|top).*(outlets?|stores?|branch\w*).*(tickets?|issues?|alerts?|actions?)|(most|highest|largest|top).*(open )?(tickets?|issues?|alerts?)",
        r"(which|what|top).*(outlets?|stores?|branch\w*).*\b(tickets?|issues?|alerts?|actions?)\b|(most|highest|largest|top).*(open )?\b(tickets?|issues?|alerts?)\b")
    rep(r'(r"(negative|bad|poor|worst|unhappy|angry|low).*(review|feedback|comment|complaint)"', r'(r"\b(negative|bad|poor|worst|unhappy|angry|low)\b.*(review|feedback|comment|complaint)"')
    rep(r'(r"(positive|best|good|happy|great).*(review|feedback|comment)"', r'(r"\b(positive|best|good|happy|great)\b.*(review|feedback|comment)"')
    # 4) intent(): optional 'no rows' message, 'I do not know' flag, readable single numbers
    a = src.index("def intent(kind, q, ids, scope=None):"); b = src.index("def goods_duty(kind, q):")
    src = src[:a] + '''def _shown(v):
    if isinstance(v, float) and not v.is_integer(): return f"{v:,.2f}".rstrip("0").rstrip(".")
    return f"{int(v):,}" if isinstance(v, (int, float)) else str(v)
def intent(kind, q, ids, scope=None):
    ql = q.lower().strip()
    for pat, msg in UNSUPPORTED.get(kind, []):
        if re.search(pat, ql): return {"answer": msg, "df": None, "sql": "", "complete": True, "idk": True}        # [tuning] declined, not answered
    for item in INTENTS.get(kind, []):
        pat, title, sql, needs_id = item[:4]; empty_msg = item[4] if len(item) > 4 else None
        if not re.search(pat, ql): continue
        if needs_id and not ids: continue
        params = (ids[0],) if needs_id else ()
        if scope:                                   # a scoped user (one outlet) only sees their own rows
            if needs_id:
                if ids[0] != scope: continue
            elif "outlet_id" in sql and " WHERE " not in sql and " ORDER BY" in sql: sql = sql.replace(" ORDER BY", " WHERE outlet_id=? ORDER BY", 1); params = (scope,)
            else: continue
        try: df = Q(sql, params)
        except Exception: continue
        if df is None or not len(df):
            if empty_msg: return {"answer": empty_msg, "df": None, "sql": sql, "complete": True, "src": "intent"}                      # [tuning] 'none' is a valid answer
            if "(IST)" in title: return {"answer": "No records: " + title.rstrip(".").replace(" (IST)", "").lower() + ": none found for today's IST date.", "df": None, "sql": sql, "complete": True, "src": "intent"}
            continue
        if needs_id: return {"answer": title.format(id=ids[0]) + f" **{_shown(df.iloc[0, 0])}**", "df": None, "sql": sql.replace("?", f"'{ids[0]}'"), "src": "intent"}
        return {"answer": title, "df": df, "sql": sql.replace("?", f"'{params[0]}'") if params else sql, "src": "intent"}
    return None

''' + src[b:]
    # 5) prepare(): declined replies, scoped users, and the 'when unsure say I do not know' gate
    rep('        if r0 is not None: r0["df"] = _mask(r0.get("df"), mask); r0["complete"] = True',
        '        if r0 is not None and r0.get("idk"): return D(r0["answer"], route="idk", sugg=["What can you do?"]), raw, None, notes, ids      # [tuning]\n        if r0 is not None: r0["df"] = _mask(r0.get("df"), mask); r0["complete"] = True')
    rep('    if res is not None:\n        res["df"] = _mask(res.get("df"), mask); res["complete"] = True',
        '''    try:                                                   # [tuning] when unsure, say 'I do not know' instead of showing an unrelated table
        if res is not None and res.get("idk"): return D(res["answer"], route="idk", sugg=["What can you do?", "Briefing"] + examples[:1]), raw, None, notes, ids
        if TUNING_GATE and not (_KQ.search(q2) or is_term_q(q2)):
            _idk = lambda: (_log_unanswered(raw), (D(IDK_TEXT + "\\n".join("- " + e for e in examples[:4]) + "\\n\\nOr say **what can you do**.", route="unanswered", sugg=["What can you do?", "Briefing"]), raw, None, notes, ids))[1]
            if res is not None and res.get("src") in ("profile", "scoped") and weak_match(q2, res): return _idk()
            if res is None:
                cand = AN.answer(kind, q2, None, dec_fn(), dom) or QA.answer(kind, q2, None, mask)
                if cand and cand.get("answer"):
                    if weak_match(q2, cand): return _idk()
                    cand["df"] = _mask(cand.get("df"), mask); return None, q2, cand, notes, ids      # vetted: hand it on so the server does not compute it twice
    except Exception as e:
        import sys; print("copilot_pro gate failed:", repr(e), file=sys.stderr)
    if res is not None:
        res["df"] = _mask(res.get("df"), mask); res["complete"] = True''')
    rep('"I could not compute an exact answer to that from the tables. I would rather say so than guess.', '"I do not know. I could not compute an exact answer to that from the tables, and I would rather say so than guess.')
    rep('"Overview": ["Briefing", "How is inventory holding up?"]}\n    else:',
        '"Overview": ["Briefing", "How is inventory holding up?"], "Money & sales": ["Which outlets are making a loss?", "How much did we sell today?", "What is the average order value?"], "Marketing & audits": ["Which campaigns are active right now?", "Which marketing channel gives the best ROI?", "Which outlets failed their audit?"], "People & customers": ["How many staff are absent today?", "Which outlets are understaffed?", "How many customers have not visited in 30 days?"], "Tickets": ["Open tickets by severity", "Which tickets are overdue?"]}   # [tuning]\n    else:')
    src += '''

# ===================================================================== [tuning] helpers appended by the Copilot fine-tune
TUNING_GATE = True      # False = show whatever the generic engine returns, even when it does not match the question
IDK_TEXT = "I do not know. I could not find a reliable way to answer that from the tables, and I would rather say so than guess a number.\\n\\nYou could try:\\n"
_profile0, _scoped0, _weak_match0 = profile, scoped, weak_match
def profile(kind, e):
    r = _profile0(kind, e)
    if r: r["src"] = "profile"
    return r
def scoped(kind, e, q):
    r = _scoped0(kind, e, q)
    if r: r["src"] = "scoped"
    return r
def _syn_words(kind):
    """word -> columns/tables it is a known synonym of (qa.py CSYN and table synonyms)."""
    w = {}
    for col, syns in QA.CSYN.items():
        for s in syns + [col.replace("_", " ")]:
            for x in re.findall(r"[a-z]{3,}", s.lower()): w.setdefault(x, set()).add(col)
    for t, (syn, idc, cols, _m) in QA.TABLES[kind].items():
        for s in syn + [t.replace("_", " ")]:
            for x in re.findall(r"[a-z]{3,}", s.lower()): w.setdefault(x, set()).add(t)
    return w
def weak_match(q, res):
    """The original test, but a question word that is a known synonym of a column or table used in the SQL counts as a match ('turnover risk' -> predicted_attrition_prob)."""
    if res.get("src") == "intent": return False                      # a pattern written for this question matched: not an accident
    q = " ".join(w for w in re.findall(r"[A-Za-z0-9-]+", q or "") if w.lower() not in STOP)      # words like 'from' also appear in every SQL text: they prove nothing
    if not _weak_match0(q, res): return False
    sql = str(res.get("sql", "")).lower(); sw = _syn_words(_kind())
    for w in re.findall(r"[a-z]{3,}", (q or "").lower()):
        if w in _SW or w in _GEN or w in _S3: continue
        for cand in {w, w[:-1] if w.endswith("s") else w}:
            if any(c in sql for c in sw.get(cand, ())): return False
    return True
'''
    open(P, "w", encoding="utf-8").write(src); print("copilot_pro.py: tuning applied")

## 3 · Admin → Copilot insights panel and the "I do not know" fallback (`views_misc.py`, `server.py`)
Run after the cells that write `views_misc.py` and `server.py`.

In [ ]:
import os, sys
APP = "/content/app"
def patch(fname, edits, append=""):
    P = os.path.join(APP, fname); src = open(P, encoding="utf-8").read()
    if "# [tuning]" in src: print(fname + ": tuning already applied, nothing to do"); return
    for old, new in edits:
        assert src.count(old) == 1, fname + " anchor not found exactly once (file differs from the reference notebook):\n" + old[:100]
        src = src.replace(old, new)
    open(P, "w", encoding="utf-8").write(src + append); print(fname + ": tuning applied")
# ---- Admin > Copilot insights: answer rate before / after
patch("views_misc.py", [(
 '        bl.append(B.table([{"route": r[0], "questions": r[1]} for r in I["routes"]]',
 '        try: bl += _tuning_blocks()                                                            # [tuning] answer rate before / after the fine-tune\n        except Exception as _e: bl.append(B.callout("Fine-tune results could not be shown: " + repr(_e)[:120], "warn"))\n        bl.append(B.table([{"route": r[0], "questions": r[1]} for r in I["routes"]]')],
 append='''

def _tuning_blocks():
    """[tuning] Answer rate before / after the Copilot fine-tune (copilot_eval.py fills the tables; run the 'Evaluate the Copilot' notebook cell)."""
    import copilot_eval as CE
    S = CE.summary(); b, a = S.get("before", {}), S.get("after", {})
    if not (a.get("ALL") and b.get("ALL")): return [B.callout("Answer rate before / after the fine-tune is not measured yet: run the 'Evaluate the Copilot' cell of the fine-tune notebook, then reload this tab.", "info")]
    pct = lambda d, k: (f"{round(100 * d[k] / d['n_answerable'])}%" if d and d["n_answerable"] else "–")
    idk = lambda d: (f"{d['idk_ok']} of {d['n_idk']}" if d and d["n_idk"] else "–")
    x, y = b["ALL"], a["ALL"]
    out = [B.kpis(("Answer rate: before", pct(x, "answered"), f"{x['answered']} of {x['n_answerable']} answerable questions"), ("Answer rate: after", pct(y, "answered"), f"{y['answered']} of {y['n_answerable']} · measured " + str(y["ts"]).replace("T", " ")),
                  ("Verified correct: before", pct(x, "correct"), "numbers match an independent SQL check"), ("Verified correct: after", pct(y, "correct"), f"{y['correct']} of {y['n_answerable']}"),
                  ("'I do not know' when it should", f"{idk(x)} → {idk(y)}", "questions the data cannot answer"))]
    names = {"TUNE": "Tuning questions (used to write the patterns)", "HOLD": "Paraphrases (written first; wording known while tuning)", "BLIND": "Blind questions (written after tuning, run once)", "ALL": "All questions"}
    rows = [{"question set": names[k], "answerable": b[k]["n_answerable"], "answer rate before": pct(b[k], "answered"), "answer rate after": pct(a[k], "answered"), "correct before": pct(b[k], "correct"), "correct after": pct(a[k], "correct"), "I do not know before": idk(b[k]), "I do not know after": idk(a[k])} for k in ("TUNE", "HOLD", "BLIND", "ALL") if b.get(k) and a.get(k)]
    out.append(B.table(rows, ["question set", "answerable", "answer rate before", "answer rate after", "correct before", "correct after", "I do not know before", "I do not know after"], title="Answer rate before and after the fine-tune", h=min(300, 60 + 34 * len(rows)), key="ci_ev"))
    bad = [{"category": i["category"], "question": i["question"], "before": i["before"], "after": i["after"], "why": i["why_after"]} for i in S["items"] if i["after"].endswith("✗")]
    out.append(B.table(bad, ["category", "question", "before", "after", "why"], title="Still not right after the fine-tune: add a pattern or a synonym for these", h=min(320, 60 + 34 * max(1, len(bad))), key="ci_bad"))
    return out
''')
# ---- server: the fallback says 'I do not know' (and the unanswered check follows the new wording)
patch("server.py", [
 ('"I could not compute an exact answer to that from the tables. What the agents responsible for this question found:', '"I do not know. I could not compute an exact answer to that from the tables, so I will not guess a number. What the agents responsible for this question found:'),
 ('elif ans_en.strip().startswith("I could not compute"):', 'elif ans_en.strip().startswith(("I do not know", "I could not compute")):      # [tuning]')])

## 4 · The test harness `copilot_eval.py`

In [ ]:
%%writefile /content/app/copilot_eval.py
"""Copilot evaluation: answer rate before / after fine-tuning (Franchise).
Replays a fixed question list through the same layers the web app uses (copilot_pro.prepare -> analyst.py -> qa.py, no GPU, no LLM) and scores each reply:
  answered  the Copilot returned an exact SQL answer that is on topic (not 'unanswered', not a weak/unrelated match)
  correct   answered AND the SQL/the numbers shown match an independent ground-truth query on the live database
  idk       for questions the data cannot answer: the Copilot declined ('I do not know') instead of inventing a number
Two question sets: TUNE (used while adding intents/synonyms) and HOLD (paraphrases written beforehand and never used for tuning, so it measures generalisation).
Run from the notebook (it starts one process on the ORIGINAL code for 'before' and one on the tuned code for 'after'); results go to tables copilot_eval_runs / copilot_eval_items and show in Admin > Copilot insights."""
import os, re, sys, json, time, sqlite3, datetime

# (id, set, category, question, expect, checks)   expect: 'answer' | 'idk'     checks: sql=[regex all found in the SQL], truth=SQL whose first value must appear in the reply, empty=True when 'none found' is the right answer
def _b(i, s, cat, q, expect="answer", **chk): return dict(id=i, set=s, cat=cat, q=q, expect=expect, chk=chk)
TODAY, YEST, W7 = "substr(ts,1,10)=date('now','localtime')", "substr(ts,1,10)=date('now','localtime','-1 day')", "substr(ts,1,10)>=date('now','localtime','-6 day')"
OPEN = "status IN ('open','proposed')"
BENCH = [
 # ---- profit and cost
 _b("t01", "TUNE", "Profit & cost", "Which outlets are making a loss?", sql=[r"operating_costs"], empty=True),
 _b("t02", "TUNE", "Profit & cost", "Which outlet has the highest profit margin?", sql=[r"operating_costs"], truth="SELECT outlet_id FROM outlets ORDER BY (revenue-operating_costs)/revenue DESC LIMIT 1"),
 _b("t03", "TUNE", "Profit & cost", "What is the profit of OUT-005?", sql=[r"operating_costs"], truth="SELECT CAST(ROUND(revenue-operating_costs,0) AS INT) FROM outlets WHERE outlet_id='OUT-005'"),
 _b("t04", "TUNE", "Profit & cost", "Show the top 5 most profitable outlets", sql=[r"operating_costs"], truth="SELECT outlet_id FROM outlets ORDER BY revenue-operating_costs DESC LIMIT 1"),
 _b("t05", "TUNE", "Profit & cost", "What is the profit margin of OUT-012?", sql=[r"operating_costs"], truth="SELECT ROUND(100.0*(revenue-operating_costs)/revenue,1) FROM outlets WHERE outlet_id='OUT-012'"),
 _b("t06", "TUNE", "Profit & cost", "Which outlets have the highest operating costs?", sql=[r"operating_costs"], truth="SELECT outlet_id FROM outlets ORDER BY operating_costs DESC LIMIT 1"),
 # ---- sales and transactions
 _b("t07", "TUNE", "Sales", "How much did we sell today?", sql=[r"\bsales\b"], truth=f"SELECT CAST(ROUND(SUM(amount),0) AS INT) FROM sales WHERE {TODAY}"),
 _b("t08", "TUNE", "Sales", "What were total sales yesterday?", sql=[r"\bsales\b"], truth=f"SELECT CAST(ROUND(SUM(amount),0) AS INT) FROM sales WHERE {YEST}"),
 _b("t09", "TUNE", "Sales", "How many transactions were there yesterday?", sql=[r"tx_count"], truth=f"SELECT SUM(tx_count) FROM sales WHERE {YEST}"),
 _b("t10", "TUNE", "Sales", "What is the average order value?", sql=[r"tx_count"], truth="SELECT ROUND(SUM(amount)/SUM(tx_count),2) FROM sales WHERE tx_count>0"),
 _b("t11", "TUNE", "Sales", "Total sales in the last 7 days", sql=[r"\bsales\b"], truth=f"SELECT CAST(ROUND(SUM(amount),0) AS INT) FROM sales WHERE {W7}"),
 _b("t12", "TUNE", "Sales", "Which outlet sold the most today?", sql=[r"\bsales\b"], truth=f"SELECT outlet_id FROM sales WHERE {TODAY} GROUP BY outlet_id ORDER BY SUM(amount) DESC LIMIT 1"),
 _b("t13", "TUNE", "Sales", "Which outlets make the most revenue per employee?", sql=[r"staff_headcount"], truth="SELECT outlet_id FROM outlets ORDER BY revenue*1.0/staff_headcount DESC LIMIT 1"),
 # ---- stock
 _b("t14", "TUNE", "Stock", "Which items are out of stock?", sql=[r"current_stock\s*<=?\s*0"], truth="SELECT outlet_id FROM inventory WHERE current_stock<=0 ORDER BY weekly_demand DESC LIMIT 1"),
 _b("t15", "TUNE", "Stock", "Which outlets have the most items below reorder level?", sql=[r"reorder_threshold"], truth="SELECT outlet_id FROM inventory WHERE current_stock<=reorder_threshold GROUP BY outlet_id ORDER BY COUNT(*) DESC, outlet_id LIMIT 1"),
 _b("t16", "TUNE", "Stock", "Show inventory by category", sql=[r"category"], truth="SELECT category FROM inventory GROUP BY category ORDER BY SUM(current_stock) DESC LIMIT 1"),
 _b("t17", "TUNE", "Stock", "Which SKUs have the highest weekly demand?", sql=[r"weekly_demand"], truth="SELECT sku_name FROM inventory ORDER BY weekly_demand DESC LIMIT 1"),
 # ---- campaigns
 _b("t18", "TUNE", "Campaigns", "Which campaigns are active right now?", sql=[r"end_date"], truth="SELECT campaign_id FROM marketing WHERE start_date<=date('now','localtime') AND end_date>=date('now','localtime') ORDER BY end_date, campaign_id LIMIT 1"),
 _b("t19", "TUNE", "Campaigns", "How many campaigns are running?", sql=[r"end_date"], truth="SELECT COUNT(*) FROM marketing WHERE start_date<=date('now','localtime') AND end_date>=date('now','localtime')"),
 _b("t20", "TUNE", "Campaigns", "Which marketing channel gives the best ROI?", sql=[r"GROUP BY channel"], truth="SELECT channel FROM marketing GROUP BY channel ORDER BY AVG(actual_roi) DESC LIMIT 1"),
 _b("t21", "TUNE", "Campaigns", "Which campaigns have already ended?", sql=[r"end_date"], truth="SELECT campaign_id FROM marketing WHERE end_date<date('now','localtime') ORDER BY end_date DESC, campaign_id LIMIT 1"),
 _b("t22", "TUNE", "Campaigns", "What is the total marketing budget?", sql=[r"budget"], truth="SELECT CAST(ROUND(SUM(budget),0) AS INT) FROM marketing"),
 # ---- audits
 _b("t23", "TUNE", "Audits", "Which outlets failed their audit?", sql=[r"audits"], truth="SELECT outlet_id FROM audits WHERE status IN ('Fail','Action Required') ORDER BY audit_date DESC, score, audit_id LIMIT 1"),
 _b("t24", "TUNE", "Audits", "What is the audit pass rate?", sql=[r"audits"], truth="SELECT ROUND(100.0*SUM(status='Pass')/COUNT(*),1) FROM audits"),
 _b("t25", "TUNE", "Audits", "Which outlets have the lowest average audit score?", sql=[r"audits"], truth="SELECT outlet_id FROM audits GROUP BY outlet_id ORDER BY AVG(score), outlet_id LIMIT 1"),
 _b("t26", "TUNE", "Audits", "Show audit violations by category", sql=[r"GROUP BY category"], truth="SELECT category FROM audits GROUP BY category ORDER BY SUM(violations) DESC, category LIMIT 1"),
 # ---- staff
 _b("t27", "TUNE", "Staff", "How many staff are absent today?", sql=[r"attendance"], truth=f"SELECT COUNT(DISTINCT staff_id) FROM attendance WHERE status='Absent' AND {TODAY}"),
 _b("t28", "TUNE", "Staff", "Which outlet has the most staff?", sql=[r"staff_headcount"], truth="SELECT outlet_id FROM outlets ORDER BY staff_headcount DESC, outlet_id LIMIT 1"),
 _b("t29", "TUNE", "Staff", "Which staff have the lowest job satisfaction?", sql=[r"job_satisfaction"]),
 _b("t30", "TUNE", "Staff", "Which outlets are short of staff?", sql=[r"staffing_shortage"]),
 _b("t31", "TUNE", "Staff", "What is the average salary by role?", sql=[r"GROUP BY role"], truth="SELECT role FROM staff GROUP BY role ORDER BY AVG(salary) DESC LIMIT 1"),
 _b("t32", "TUNE", "Staff", "Who has the longest tenure?", sql=[r"tenure_years"]),
 _b("t33", "TUNE", "Staff", "How many cooks do we have?", sql=[r"role"], truth="SELECT COUNT(*) FROM staff WHERE role='Cook'"),
 _b("t34", "TUNE", "Staff", "Staff with more than 30 overtime hours", sql=[r"overtime_hrs\s*>"], ),
 _b("t35", "TUNE", "Staff", "Which staff are most likely to quit?", sql=[r"predicted_attrition_prob"], truth="SELECT staff_id FROM staff ORDER BY predicted_attrition_prob DESC LIMIT 1"),
 # ---- customers
 _b("t36", "TUNE", "Customers", "How many customers have not visited in over 30 days?", sql=[r"customers_fr"], truth="SELECT COUNT(*) FROM customers_fr WHERE days_since_last_visit>30"),
 _b("t37", "TUNE", "Customers", "Which outlets have the most lapsed customers?", sql=[r"customers_fr"], truth="SELECT outlet_id FROM customers_fr WHERE days_since_last_visit>30 GROUP BY outlet_id ORDER BY COUNT(*) DESC, outlet_id LIMIT 1"),
 _b("t38", "TUNE", "Customers", "Who are our top spending customers?", sql=[r"total_spend_90d"], truth="SELECT customer_id FROM customers_fr ORDER BY total_spend_90d DESC, customer_id LIMIT 1"),
 _b("t39", "TUNE", "Customers", "What is the average customer spend?", sql=[r"total_spend_90d"], truth="SELECT ROUND(AVG(total_spend_90d),2) FROM customers_fr"),
 _b("t40", "TUNE", "Customers", "How many customers do we have?", sql=[r"customers_fr"], truth="SELECT COUNT(*) FROM customers_fr"),
 # ---- feedback
 _b("t41", "TUNE", "Feedback", "Show reviews by star rating", sql=[r"GROUP BY rating"], truth="SELECT COUNT(*) FROM feedback WHERE rating=5"),
 _b("t42", "TUNE", "Feedback", "Which outlets have the lowest customer ratings?", sql=[r"feedback"], truth="SELECT outlet_id FROM feedback GROUP BY outlet_id HAVING COUNT(*)>=3 ORDER BY AVG(rating), outlet_id LIMIT 1"),
 _b("t43", "TUNE", "Feedback", "What are customers complaining about the most?", sql=[r"feedback"]),
 _b("t44", "TUNE", "Feedback", "Which outlets are below 3.5 customer satisfaction?", sql=[r"customer_satisfaction\s*<"], truth="SELECT COUNT(*) FROM outlets WHERE customer_satisfaction<3.5"),
 # ---- tickets
 _b("t45", "TUNE", "Tickets", "Open tickets by severity", sql=[r"GROUP BY severity"], truth=f"SELECT COUNT(*) FROM agent_actions WHERE {OPEN} AND severity='Critical'"),
 _b("t46", "TUNE", "Tickets", "Which tickets are overdue?", sql=[r"\bdue\b"], truth=f"SELECT action_id FROM agent_actions WHERE {OPEN} AND due<strftime('%Y-%m-%dT%H:%M:%S','now','localtime') ORDER BY due, action_id LIMIT 1"),
 _b("t47", "TUNE", "Tickets", "What is awaiting approval?", sql=[r"proposed"]),
 _b("t48", "TUNE", "Tickets", "How many critical tickets are open?", sql=[r"agent_actions"], truth=f"SELECT COUNT(*) FROM agent_actions WHERE {OPEN} AND severity='Critical'"),
 _b("t49", "TUNE", "Tickets", "Tickets by type", sql=[r"GROUP BY type"], truth=f"SELECT type FROM agent_actions WHERE {OPEN} GROUP BY type ORDER BY COUNT(*) DESC, type LIMIT 1"),
 # ---- outlets
 _b("t50", "TUNE", "Outlets", "Which city has the highest revenue?", sql=[r"GROUP BY location"], truth="SELECT location FROM outlets GROUP BY location ORDER BY SUM(revenue) DESC LIMIT 1"),
 _b("t51", "TUNE", "Outlets", "Which tier performs best?", sql=[r"GROUP BY tier"], truth="SELECT tier FROM outlets GROUP BY tier ORDER BY SUM(revenue) DESC LIMIT 1"),
 _b("t52", "TUNE", "Outlets", "How many outlets are in Mumbai?", sql=[r"outlets"], truth="SELECT COUNT(*) FROM outlets WHERE location='Mumbai'"),
 _b("t53", "TUNE", "Outlets", "Top 5 outlets by revenue", sql=[r"revenue"], truth="SELECT outlet_id FROM outlets ORDER BY revenue DESC LIMIT 1"),
 _b("t54", "TUNE", "Outlets", "Which outlets have the lowest customer satisfaction?", sql=[r"customer_satisfaction"], truth="SELECT outlet_id FROM outlets ORDER BY customer_satisfaction LIMIT 1"),
 _b("t55", "TUNE", "Outlets", "Which outlets have the most open tickets?", sql=[r"agent_actions"], truth=f"SELECT target FROM agent_actions WHERE {OPEN} AND target LIKE 'OUT-%' GROUP BY target ORDER BY COUNT(*) DESC, SUM(severity='Critical') DESC LIMIT 1"),
 _b("t56", "TUNE", "Outlets", "Overtime hours by outlet", sql=[r"overtime"], truth="SELECT outlet_id FROM staff GROUP BY outlet_id ORDER BY SUM(overtime_hrs) DESC LIMIT 1"),
 # ---- out of scope: the data cannot answer, the right reply is 'I do not know'
 _b("i01", "TUNE", "I do not know", "What is the weather in Delhi today?", "idk"),
 _b("i02", "TUNE", "I do not know", "What are our competitors charging?", "idk"),
 _b("i03", "TUNE", "I do not know", "Who is our cheapest supplier?", "idk"),
 _b("i04", "TUNE", "I do not know", "What is the best selling menu item?", "idk"),
 _b("i05", "TUNE", "I do not know", "What is the gender ratio of our staff?", "idk"),
 _b("i06", "TUNE", "I do not know", "What is the expiry date of milk?", "idk"),
 _b("i07", "TUNE", "I do not know", "How much GST did we pay last month?", "idk"),
 _b("i08", "TUNE", "I do not know", "What will our revenue be next year?", "idk"),
 _b("i09", "TUNE", "I do not know", "What is the phone number of STF-0012?", "idk"),
 _b("i10", "TUNE", "I do not know", "How much rent does OUT-004 pay?", "idk"),
 _b("i11", "TUNE", "I do not know", "How many Zomato orders did we get?", "idk"),
 _b("i12", "TUNE", "I do not know", "Which employees deserve a promotion?", "idk"),
 _b("i13", "TUNE", "I do not know", "How many units of chicken did we sell yesterday?", "idk"),
 _b("i14", "TUNE", "I do not know", "Which outlet has the best parking space?", "idk"),
 _b("i15", "TUNE", "I do not know", "How many chairs are there in OUT-010?", "idk"),
 # ================= held-out set: written before any tuning, paraphrases of the same needs, never used to design a pattern =================
 _b("h01", "HOLD", "Profit & cost", "Are any of our outlets operating at a loss?", sql=[r"operating_costs"], empty=True),
 _b("h02", "HOLD", "Profit & cost", "Rank the outlets by profit", sql=[r"operating_costs"], truth="SELECT outlet_id FROM outlets ORDER BY revenue-operating_costs DESC LIMIT 1"),
 _b("h03", "HOLD", "Profit & cost", "What is the profit for OUT-020?", sql=[r"operating_costs"], truth="SELECT CAST(ROUND(revenue-operating_costs,0) AS INT) FROM outlets WHERE outlet_id='OUT-020'"),
 _b("h04", "HOLD", "Sales", "How much revenue did we make today?", sql=[r"\bsales\b"], truth=f"SELECT CAST(ROUND(SUM(amount),0) AS INT) FROM sales WHERE {TODAY}"),
 _b("h05", "HOLD", "Sales", "How much did we earn in the past 7 days?", sql=[r"\bsales\b"], truth=f"SELECT CAST(ROUND(SUM(amount),0) AS INT) FROM sales WHERE {W7}"),
 _b("h06", "HOLD", "Sales", "How many bills were generated yesterday?", sql=[r"tx_count"], truth=f"SELECT SUM(tx_count) FROM sales WHERE {YEST}"),
 _b("h07", "HOLD", "Sales", "What is the typical basket size?", sql=[r"tx_count"], truth="SELECT ROUND(SUM(amount)/SUM(tx_count),2) FROM sales WHERE tx_count>0"),
 _b("h08", "HOLD", "Stock", "Show me the SKUs that have sold out", sql=[r"current_stock\s*<=?\s*0"], truth="SELECT outlet_id FROM inventory WHERE current_stock<=0 ORDER BY weekly_demand DESC LIMIT 1"),
 _b("h09", "HOLD", "Stock", "Which branches have the most low-stock items?", sql=[r"reorder_threshold"], truth="SELECT outlet_id FROM inventory WHERE current_stock<=reorder_threshold GROUP BY outlet_id ORDER BY COUNT(*) DESC, outlet_id LIMIT 1"),
 _b("h10", "HOLD", "Campaigns", "Show me the promotions running at the moment", sql=[r"end_date"], truth="SELECT campaign_id FROM marketing WHERE start_date<=date('now','localtime') AND end_date>=date('now','localtime') ORDER BY end_date, campaign_id LIMIT 1"),
 _b("h11", "HOLD", "Campaigns", "Which promotions have expired?", sql=[r"end_date"], truth="SELECT campaign_id FROM marketing WHERE end_date<date('now','localtime') ORDER BY end_date DESC, campaign_id LIMIT 1"),
 _b("h12", "HOLD", "Campaigns", "Which advertising platform performs best on return?", sql=[r"GROUP BY channel"], truth="SELECT channel FROM marketing GROUP BY channel ORDER BY AVG(actual_roi) DESC LIMIT 1"),
 _b("h13", "HOLD", "Audits", "Which branches did not pass their inspection?", sql=[r"audits"], truth="SELECT outlet_id FROM audits WHERE status IN ('Fail','Action Required') ORDER BY audit_date DESC, score, audit_id LIMIT 1"),
 _b("h14", "HOLD", "Audits", "What percentage of audits passed?", sql=[r"audits"], truth="SELECT ROUND(100.0*SUM(status='Pass')/COUNT(*),1) FROM audits"),
 _b("h15", "HOLD", "Staff", "How many employees were absent today?", sql=[r"attendance"], truth=f"SELECT COUNT(DISTINCT staff_id) FROM attendance WHERE status='Absent' AND {TODAY}"),
 _b("h16", "HOLD", "Staff", "Which outlet employs the most people?", sql=[r"staff_headcount"], truth="SELECT outlet_id FROM outlets ORDER BY staff_headcount DESC, outlet_id LIMIT 1"),
 _b("h17", "HOLD", "Staff", "Which employees seem unhappy at work?", sql=[r"job_satisfaction"]),
 _b("h18", "HOLD", "Staff", "Which outlets are understaffed?", sql=[r"staffing_shortage"]),
 _b("h19", "HOLD", "Customers", "How many guests have not come back in a month?", sql=[r"customers_fr"], truth="SELECT COUNT(*) FROM customers_fr WHERE days_since_last_visit>30"),
 _b("h20", "HOLD", "Customers", "Which outlets have the most inactive customers?", sql=[r"customers_fr"], truth="SELECT outlet_id FROM customers_fr WHERE days_since_last_visit>30 GROUP BY outlet_id ORDER BY COUNT(*) DESC, outlet_id LIMIT 1"),
 _b("h21", "HOLD", "Customers", "Who are the highest spending shoppers?", sql=[r"total_spend_90d"], truth="SELECT customer_id FROM customers_fr ORDER BY total_spend_90d DESC, customer_id LIMIT 1"),
 _b("h22", "HOLD", "Feedback", "Which stores have the best average star rating?", sql=[r"feedback"], truth="SELECT outlet_id FROM feedback GROUP BY outlet_id HAVING COUNT(*)>=3 ORDER BY AVG(rating) DESC, outlet_id LIMIT 1"),
 _b("h23", "HOLD", "Tickets", "Break down the open tickets by severity please", sql=[r"GROUP BY severity"], truth=f"SELECT COUNT(*) FROM agent_actions WHERE {OPEN} AND severity='Critical'"),
 _b("h24", "HOLD", "Tickets", "Any overdue tasks?", sql=[r"\bdue\b"], truth=f"SELECT action_id FROM agent_actions WHERE {OPEN} AND due<strftime('%Y-%m-%dT%H:%M:%S','now','localtime') ORDER BY due, action_id LIMIT 1"),
 _b("h25", "HOLD", "Tickets", "Which actions need my approval?", sql=[r"proposed"]),
 _b("h26", "HOLD", "Outlets", "Which town has the highest sales?", sql=[r"GROUP BY location"], truth="SELECT location FROM outlets GROUP BY location ORDER BY SUM(revenue) DESC LIMIT 1"),
 _b("h27", "HOLD", "Outlets", "Which format of store performs worst?", sql=[r"GROUP BY tier"], truth="SELECT tier FROM outlets GROUP BY tier ORDER BY SUM(revenue) LIMIT 1"),
 _b("h28", "HOLD", "Typos", "Which outlets are makin a loss?", sql=[r"operating_costs"], empty=True),
 _b("h29", "HOLD", "Typos", "Show the profit by outlte", sql=[r"operating_costs"], truth="SELECT outlet_id FROM outlets ORDER BY revenue-operating_costs DESC LIMIT 1"),
 _b("h30", "HOLD", "Typos", "How many staf were absent today?", sql=[r"attendance"], truth=f"SELECT COUNT(DISTINCT staff_id) FROM attendance WHERE status='Absent' AND {TODAY}"),
 _b("hi1", "HOLD", "I do not know", "What's the temperature in Mumbai right now?", "idk"),
 _b("hi2", "HOLD", "I do not know", "How do our prices compare with rival chains?", "idk"),
 _b("hi3", "HOLD", "I do not know", "What is our most popular dish?", "idk"),
 _b("hi4", "HOLD", "I do not know", "Which vendor delivers late the most?", "idk"),
 _b("hi5", "HOLD", "I do not know", "What is the average age of our customers?", "idk"),
 _b("hi6", "HOLD", "I do not know", "What is the monthly electricity bill of OUT-003?", "idk"),
 _b("hi7", "HOLD", "I do not know", "Which outlet has the best view from the dining area?", "idk"),
 _b("hi8", "HOLD", "I do not know", "How many tables does OUT-010 have?", "idk"),
 # ================= BLIND set: written AFTER the tuning was frozen and run once, with no changes made afterwards =================
 _b("b01", "BLIND", "Profit & cost", "Which stores lost money?", sql=[r"operating_costs"], empty=True),
 _b("b02", "BLIND", "Outlets", "What is the total revenue across all outlets?", sql=[r"SUM\(revenue\)"], truth="SELECT CAST(ROUND(SUM(revenue),0) AS INT) FROM outlets"),
 _b("b03", "BLIND", "Sales", "How much did the stores bring in yesterday?", sql=[r"\bsales\b"], truth=f"SELECT CAST(ROUND(SUM(amount),0) AS INT) FROM sales WHERE {YEST}"),
 _b("b04", "BLIND", "Sales", "Number of transactions today", sql=[r"tx_count"], truth=f"SELECT SUM(tx_count) FROM sales WHERE {TODAY}"),
 _b("b05", "BLIND", "Profit & cost", "Which branch has the lowest profit?", sql=[r"operating_costs"], truth="SELECT outlet_id FROM outlets ORDER BY revenue-operating_costs LIMIT 1"),
 _b("b06", "BLIND", "Stock", "Give me the SKUs with zero stock", sql=[r"current_stock\s*<=?\s*0"], truth="SELECT outlet_id FROM inventory WHERE current_stock<=0 ORDER BY weekly_demand DESC LIMIT 1"),
 _b("b07", "BLIND", "Campaigns", "Campaigns that are currently live", sql=[r"end_date"], truth="SELECT campaign_id FROM marketing WHERE start_date<=date('now','localtime') AND end_date>=date('now','localtime') ORDER BY end_date, campaign_id LIMIT 1"),
 _b("b08", "BLIND", "Campaigns", "Which marketing channel has the weakest return?", sql=[r"GROUP BY channel"], truth="SELECT channel FROM marketing GROUP BY channel ORDER BY AVG(actual_roi) LIMIT 1"),
 _b("b09", "BLIND", "Audits", "List the audits that failed", sql=[r"audits"], truth="SELECT audit_id FROM audits WHERE status IN ('Fail','Action Required') ORDER BY audit_date DESC, score, audit_id LIMIT 1"),
 _b("b10", "BLIND", "Audits", "How many audits did we do?", sql=[r"audits"], truth="SELECT COUNT(*) FROM audits"),
 _b("b11", "BLIND", "Staff", "Which employees are absent today?", sql=[r"attendance"]),
 _b("b12", "BLIND", "Staff", "Which outlet has the fewest employees?", sql=[r"staff_headcount"], truth="SELECT outlet_id FROM outlets ORDER BY staff_headcount, outlet_id LIMIT 1"),
 _b("b13", "BLIND", "Staff", "Which staff have the highest attrition risk?", sql=[r"predicted_attrition_prob"], truth="SELECT staff_id FROM staff ORDER BY predicted_attrition_prob DESC LIMIT 1"),
 _b("b14", "BLIND", "Feedback", "What is the average review rating?", sql=[r"AVG\(rating\)"], truth="SELECT ROUND(AVG(rating),2) FROM feedback"),
 _b("b15", "BLIND", "Feedback", "Which outlet got the best reviews?", sql=[r"feedback"], truth="SELECT outlet_id FROM feedback GROUP BY outlet_id HAVING COUNT(*)>=3 ORDER BY AVG(rating) DESC, outlet_id LIMIT 1"),
 _b("b16", "BLIND", "Tickets", "How many critical alerts are open right now?", sql=[r"agent_actions"], truth=f"SELECT COUNT(*) FROM agent_actions WHERE {OPEN} AND severity='Critical'"),
 _b("b17", "BLIND", "Tickets", "Show open tickets by outlet", sql=[r"agent_actions"], truth=f"SELECT target FROM agent_actions WHERE {OPEN} AND target LIKE 'OUT-%' GROUP BY target ORDER BY COUNT(*) DESC, SUM(severity='Critical') DESC LIMIT 1"),
 _b("b18", "BLIND", "Outlets", "Which city has the lowest revenue?", sql=[r"GROUP BY location"], truth="SELECT location FROM outlets GROUP BY location ORDER BY SUM(revenue) LIMIT 1"),
 _b("b19", "BLIND", "Staff", "Average salary of store managers", sql=[r"salary"], truth="SELECT ROUND(AVG(salary),0) FROM staff WHERE role='Store Manager'"),
 _b("b20", "BLIND", "Outlets", "How many outlets does Pune have?", sql=[r"outlets"], truth="SELECT COUNT(*) FROM outlets WHERE location='Pune'"),
 _b("b21", "BLIND", "Outlets", "Tell me about OUT-010", sql=[r"outlets"], truth="SELECT outlet_name FROM outlets WHERE outlet_id='OUT-010'"),
 _b("b22", "BLIND", "Outlets", "Compare OUT-001 and OUT-002", sql=[r"outlets"]),
 _b("b23", "BLIND", "Outlets", "Top 10 outlets by customer satisfaction", sql=[r"customer_satisfaction"], truth="SELECT outlet_id FROM outlets ORDER BY customer_satisfaction DESC LIMIT 1"),
 _b("bi1", "BLIND", "I do not know", "How many parking spots does OUT-005 have?", "idk"),
 _b("bi2", "BLIND", "I do not know", "What is the CEO's name?", "idk"),
 _b("bi3", "BLIND", "I do not know", "Which outlets sell vegan food?", "idk"),
 _b("bi4", "BLIND", "I do not know", "How did our stock price move today?", "idk"),
 _b("bi5", "BLIND", "I do not know", "What is the annual salary budget for next year?", "idk"),
 _b("bi6", "BLIND", "I do not know", "Which staff member is the most hardworking?", "idk"),
 _b("bi7", "BLIND", "I do not know", "How many complaints did Zomato customers raise?", "idk"),
 _b("bi8", "BLIND", "I do not know", "What colour are the uniforms at OUT-020?", "idk"),
]

EX = ["Which outlets are at highest risk?", "Why is OUT-007 flagged?", "How is inventory holding up?", "Top 5 outlets by revenue"]

def _boot(app_dir):
    sys.path.insert(0, app_dir); os.chdir(app_dir)
    global QA, AN, CP, dom, Q
    import qa as QA, analyst as AN, copilot_pro as CP, domain_franchise as dom
    from blocks import Q
    CP._log_unanswered = lambda q: None                  # the test questions must not land in the admin's unanswered backlog
def _run(q, sid):
    """Same order the server uses: copilot_pro.prepare -> exact intent result, else analyst.py, else qa.py. Returns (route, text, sql, df)."""
    direct, q2, res, notes, ids = CP.prepare("Franchise", q, sid, dom, lambda: None, EX)
    if direct: return (direct.get("route") if direct.get("route") in ("idk", "unanswered") else "chat"), direct.get("text", ""), direct.get("sql", ""), None
    try:
        if res is None: res = CP.goods_duty("Franchise", q2) or AN.answer("Franchise", q2, None, None, dom) or QA.answer("Franchise", q2, None, None)
    except Exception as e: return "error", repr(e)[:120], "", None
    if not res or not res.get("answer"): return "unanswered", "", "", None
    df = res.get("df")
    if res.get("idk") or (not res.get("sql") and df is None): return "idk", res["answer"], res.get("sql", ""), None      # a reply that runs no query and gives no table is a decline
    if CP.weak_match(q, res): return "weak", res["answer"], res.get("sql", ""), df
    return "exact", res["answer"], res.get("sql", ""), df
def _cands(v):
    try: x = float(v)
    except Exception: return [str(v)]
    out = {f"{x:,.0f}", f"{x:,.2f}", f"{x:.2f}", f"{x:.1f}", f"{x:.0f}", f"{x:,.1f}", f"{x:.1f}".rstrip("0").rstrip("."), f"{x:.2f}".rstrip("0").rstrip(".")}
    return [c for c in out if c]
def _hay(text, df):
    h = text or ""
    if df is not None:
        try: h += "\n" + QA.md(df) + "\n" + df.to_csv(index=False)
        except Exception: pass
    return h
def _check(item, route, text, sql, df):
    c = item["chk"]; hay = _hay(text, df)
    if route != "exact": return False, "route=" + route
    for rx in c.get("sql", []):
        if not re.search(rx, sql or "", re.I): return False, "SQL does not use " + rx
    if c.get("empty"):
        return (bool(re.search(r"\b(no |none|zero|0 )", hay.lower())) or df is None or not len(df)), "expected an empty result"
    if c.get("truth"):
        try: t = Q(c["truth"])
        except Exception as e: return False, "truth query failed: " + repr(e)[:60]
        if t is None or not len(t): return (bool(re.search(r"\b(no |none|nothing)", hay.lower())), "truth is empty: the reply should say none")
        v = t.iloc[0, 0]
        if v is None: return False, "truth is NULL"
        for cand in _cands(v):
            if re.search(r"(?<![\d.,])" + re.escape(cand) + r"(?![\d])", hay): return True, ""
        return False, f"expected {_cands(v)[0]} in the reply"
    return True, ""
def _db():
    import config; c = sqlite3.connect(config.DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;")
    c.execute("CREATE TABLE IF NOT EXISTS copilot_eval_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, label TEXT, set_name TEXT, n_answerable INT, answered INT, correct INT, n_idk INT, idk_ok INT, wrong_confident INT, ms REAL)")
    c.execute("CREATE TABLE IF NOT EXISTS copilot_eval_items(run_id INT, set_name TEXT, category TEXT, q TEXT, expect TEXT, route TEXT, ok INT, why TEXT, reply TEXT)")
    return c
def evaluate(label, app_dir):
    _boot(app_dir); import hashlib
    rows = []
    for k, it in enumerate(BENCH):
        t0 = time.time(); route, text, sql, df = _run(it["q"], f"eval-{label}-{k}")
        if it["expect"] == "idk": ok, why = (route in ("idk", "unanswered")), ("" if route in ("idk", "unanswered") else f"answered with route={route} instead of declining")
        else: ok, why = _check(it, route, text, sql, df)
        rows.append((it, route, ok, why, (text or "").replace("\n", " ")[:160], (time.time() - t0) * 1000))
    ts = datetime.datetime.now().isoformat(timespec="seconds")
    with _db() as c:
        for s in ("TUNE", "HOLD", "BLIND", "ALL"):
            R = [r for r in rows if s == "ALL" or r[0]["set"] == s]; A = [r for r in R if r[0]["expect"] == "answer"]; I = [r for r in R if r[0]["expect"] == "idk"]
            rid = c.execute("INSERT INTO copilot_eval_runs(ts,label,set_name,n_answerable,answered,correct,n_idk,idk_ok,wrong_confident,ms) VALUES(?,?,?,?,?,?,?,?,?,?)",
                (ts, label, s, len(A), sum(1 for r in A if r[1] == "exact"), sum(1 for r in A if r[2]), len(I), sum(1 for r in I if r[2]), sum(1 for r in I if r[1] in ("exact", "weak")), round(sum(r[5] for r in R) / max(1, len(R)), 1))).lastrowid
            if s == "ALL":
                for it, route, ok, why, reply, ms in rows: c.execute("INSERT INTO copilot_eval_items VALUES(?,?,?,?,?,?,?,?,?)", (rid, it["set"], it["cat"], it["q"], it["expect"], route, int(ok), why, reply))
    return rows
def summary():
    """For Admin > Copilot insights: the latest 'before' and 'after' run, per question set, plus the questions that still fail."""
    try:
        with _db() as c:
            def latest(label, s):
                r = c.execute("SELECT run_id, ts, n_answerable, answered, correct, n_idk, idk_ok, wrong_confident FROM copilot_eval_runs WHERE label=? AND set_name=? ORDER BY run_id DESC LIMIT 1", (label, s)).fetchone()
                return dict(zip(("run_id", "ts", "n_answerable", "answered", "correct", "n_idk", "idk_ok", "wrong_confident"), r)) if r else None
            out = {"before": {s: latest("before", s) for s in ("TUNE", "HOLD", "BLIND", "ALL")}, "after": {s: latest("after", s) for s in ("TUNE", "HOLD", "BLIND", "ALL")}}
            a = out["after"]["ALL"]; b = out["before"]["ALL"]
            out["items"] = []
            if a and b:
                for cat, q, ex, rb, okb, ra, oka, why, rep in c.execute("SELECT a.category, a.q, a.expect, b.route, b.ok, a.route, a.ok, a.why, a.reply FROM copilot_eval_items a JOIN copilot_eval_items b ON a.q=b.q AND b.run_id=? WHERE a.run_id=? ORDER BY a.rowid", (b["run_id"], a["run_id"])).fetchall():
                    out["items"].append({"category": cat, "question": q, "expect": ex, "before": rb + (" ✓" if okb else " ✗"), "after": ra + (" ✓" if oka else " ✗"), "why_after": why, "reply": rep})
            return out
    except Exception: return {"before": {}, "after": {}, "items": []}
if __name__ == "__main__":
    lab, app = sys.argv[1], sys.argv[2]
    R = evaluate(lab, app)
    for s in ("TUNE", "HOLD", "BLIND"):
        A = [r for r in R if r[0]["set"] == s and r[0]["expect"] == "answer"]; I = [r for r in R if r[0]["set"] == s and r[0]["expect"] == "idk"]
        print(f"{lab:6} {s}: answered {sum(1 for r in A if r[1]=='exact')}/{len(A)}  correct {sum(1 for r in A if r[2])}/{len(A)}  idk-ok {sum(1 for r in I if r[2])}/{len(I)}")
    if "-v" in sys.argv:
        for it, route, ok, why, reply, ms in R:
            if not ok and (len(sys.argv) < 5 or it["set"] in sys.argv[4:]): print(f"  X {it['id']} [{route}] {it['q']}  -> {why} | {reply[:90]}")

## 5 · Start the app (existing cell of the main notebook), then evaluate

In [ ]:
# RUN THIS AFTER the "Start the app" cell (the database must exist). It measures the Copilot twice on the SAME live database:
#   before = the original qa.py + copilot_pro.py (saved by the patch cells in /content/app/_orig), after = the tuned files.
# Results are stored in the database and appear in Admin > Copilot insights (reload that tab).
import os, sys, shutil, subprocess
APP, BEFORE = "/content/app", "/content/app_before"
shutil.rmtree(BEFORE, ignore_errors=True)
shutil.copytree(APP, BEFORE, ignore=shutil.ignore_patterns("static", "_orig", "__pycache__"))
for f in ("qa.py", "copilot_pro.py"): shutil.copy(f"{APP}/_orig/{f}", f"{BEFORE}/{f}")
shutil.copy(f"{APP}/copilot_eval.py", f"{BEFORE}/copilot_eval.py")
env = dict(os.environ, APP_KIND="Franchise")
for label, d in (("before", BEFORE), ("after", APP)):
    r = subprocess.run([sys.executable, "copilot_eval.py", label, d, "-v"], cwd=d, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-1500:])

## Notes for whoever integrates this

**What changed in which file**
- `qa.py`: synonym dictionaries `TUNED_COLUMN_SYNONYMS`, `TUNED_TABLE_SYNONYMS`, `TUNED_TABLES` appended at the bottom; `customer satisfaction/rating/feedback` no longer selects the customers table; "open tickets" now means open **and** awaiting approval everywhere (before, `qa.py` counted only `status='open'`, so it disagreed with the briefing: 43 vs 144 critical).
- `copilot_pro.py`: `TUNED_INTENTS` and `TUNED_UNSUPPORTED` (same tuple format as `INTENTS`; add a line to teach a new question; an optional 5th item is the message to show when the query returns no rows); `intent()` supports that and flags declined questions; `prepare()` has the "I do not know" gate (switch it off with `TUNING_GATE = False`); `weak_match` also counts column/table synonyms. Two original regexes were fixed (`satisfaction` matched `actions?`, `below` matched `low`), and "expired campaigns" no longer triggers the "no expiry dates" refusal.
- `views_misc.py`: the before/after panel in Admin → Copilot insights. `server.py`: the fallback now starts with "I do not know" and the unanswered check accepts the new wording.
- New file `copilot_eval.py`; tables `copilot_eval_runs` and `copilot_eval_items` are created in the app database.

**Check after integrating**: reload Admin → Copilot insights; ask "Which outlets are making a loss?", "How much did we sell today?", "What is the weather in Delhi?". Scoped users (one outlet) never get the company-wide intents: the existing scope rule skips any intent that is not outlet-filterable.

**Known gaps (still failing in my run, good next patterns)**: "Which stores lost money?", "How much did the stores bring in yesterday?", "Show open tickets by outlet" (phrasing the patterns do not cover); "average age of our customers", "stock price", "annual salary budget for next year", "most hardworking staff" (answered with an unrelated table instead of "I do not know"). Lapsed-customer questions only cover the 30-day threshold; other thresholds go to the generic engine. Counts such as absentees today are among the attendance rows logged today. Staff headcount shows the declared `outlets.staff_headcount` beside the number of staff rows, which differ in this data.

**Limits of my testing**: I ran the real code from your notebook on a simulated database (the sandbox had no GPU, no LLM and no web server), so the LLM path, the translated languages and the live UI were not exercised. The evaluation covers the SQL-only layers.